# Dependencies

In [3]:
import os
import polars as pl
import glob
import json
import pandas as pd
import numpy as np
import math
import joblib

## Pre-Processing

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Helper

In [4]:
def get_all_file_names(source_folder_name:str,file_format:str = ""):
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format)>0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)


In [5]:
def filter_file_names(list_file_names:list, filter_word:str):
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

# Pre-Processing

## Change CSV to Parquet

In [6]:
CHUNK_SIZE = 100000
LABEL_COLUMN = "Label"

In [7]:
COLUMNS_TO_DROP = {
    "flow id", "src ip", "source ip", "src port", "source port",
    "dst ip", "destination ip", "timestamp",
}

In [8]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [9]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [10]:
def dataframe_to_parquet(chunk: pd.DataFrame, target_folder_name: str, file_name: str, template_columns: list[str] | None = None):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

> Bisa dibuat lebih efisien create_template_columns nya

In [11]:
def create_template_columns(files: list, source_folder_name: str):
    all_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        for c in cols:
            if c not in seen:
                seen.add(c)
                all_columns.append(c)

    template_columns = []
    for c in all_columns:
        if c not in COLUMNS_TO_DROP:
            template_columns.append(c)
    return template_columns

In [12]:
def convert_all_file_to_parquet(
    source_folder_name: str,
    target_folder_name: str,
    chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files,source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    for i, file_name in enumerate(csv_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        base_name = os.path.splitext(file_name)[0]

        for chunk_number, chunk in enumerate(pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1):
            dataframe_to_parquet(chunk, target_folder_name, f"{base_name}_{chunk_number:05d}.parquet", template_columns)

    print(f"\nFinished processing {total} files.")

In [ ]:
CSV_FOLDER_NAME = "cse-cic-ids2018"
RAW_DATA_FOLDER_NAME = "parquet-raw-chunks"

In [14]:
convert_all_file_to_parquet(CSV_FOLDER_NAME,RAW_DATA_FOLDER_NAME,CHUNK_SIZE)

Processing [10/10] 2018-03-02-Friday_TrafficForML_CICFlowMeter.csv.....
Finished processing 10 files.


## Exploratory Data Analysis

## Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [15]:
def calculate_inf_values(
    source_folder_name: str
):
    parquet_files = get_all_file_names(source_folder_name,"parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [16]:
calculate_inf_values(RAW_DATA_FOLDER_NAME)

Processing [168/168] 2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......

Completed. Processed 168 files.
Found 131799 inf values


In [17]:
def change_inf_to_nan(
    source_folder_name: str,
    target_folder_name: str
):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name,"parquet")

    total = len(parquet_files)

    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        target_file = os.path.join(target_folder_name, file_name)

        df = pd.read_parquet(source_file)

        df.replace([np.inf, -np.inf],np.nan,inplace=True)
        df.to_parquet(target_file,index=False)

    print(f"\nCompleted. Processed {total} files.")

In [ ]:
NO_INF_DATA_FOLDER_NAME = "parquet-without-infinite-values"

In [19]:
change_inf_to_nan(RAW_DATA_FOLDER_NAME, NO_INF_DATA_FOLDER_NAME)

Processing [168/168] 2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......
Completed. Processed 168 files.


In [20]:
calculate_inf_values(NO_INF_DATA_FOLDER_NAME)

Processing [168/168] 2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......

Completed. Processed 168 files.
Found 0 inf values


The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

```text
Completed. Processed 168 files.
Found 131799 inf values
```

**After cleaning:**

```text
Completed. Processed 168 files.
Found 0.0 inf values
```

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [21]:
def create_train_dev_test_folder(source_folder_name:str, target_folder_name:str,target_day):

    parquet_files = get_all_file_names(source_folder_name,"parquet")
    parquet_files = filter_file_names(parquet_files,target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    day_folder = os.path.join(
        target_folder_name,
        f"{target_day}"
    )

    train_folder = os.path.join(day_folder, "train")
    dev_folder = os.path.join(day_folder, "dev")
    test_folder = os.path.join(day_folder, "test")

    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(dev_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)

    return parquet_files, train_folder,dev_folder,test_folder

In [22]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    train_folder: str,
    dev_folder: str,
    test_folder: str,
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)

        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)

        train_df, temp_df = train_test_split(
            df,
            test_size=dev_size + test_size,
            random_state=random_state,
            stratify=df[label_column],
        )
        dev_df, test_df = train_test_split(
            temp_df,
            test_size=test_size / (dev_size + test_size),
            random_state=random_state,
            stratify=temp_df[label_column],
        )

        for split_df, folder in (
            (train_df, train_folder),
            (dev_df, dev_folder),
            (test_df, test_folder),
        ):
            split_df.to_parquet(os.path.join(folder, file_name), index=False)

    print(f"\nCompleted splitting {total} files.")

In [23]:
def split_a_single_day(
    source_folder_name: str,
    target_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, train_folder, dev_folder, test_folder = create_train_dev_test_folder(
        source_folder_name, target_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        train_folder,
        dev_folder,
        test_folder,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

In [24]:
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [25]:
SPLIT_DATA_FOLDER_NAME = "parquet-split"

In [26]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,bruteforce)

Found 11 files for 2018-02-14
Processing [11/11] 2018-02-14-Wednesday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [27]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,dos_golden)

Found 11 files for 2018-02-15
Processing [11/11] 2018-02-15-Thursday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [28]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,dos_hulk)

Found 11 files for 2018-02-16
Processing [11/11] 2018-02-16-Friday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [29]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,ddos_http)

Found 80 files for 2018-02-20
Processing [80/80] 2018-02-20-Tuesday_TrafficForML_CICFlowMeter_00080.parquet...
Completed splitting 80 files.


In [30]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,ddos_udp)

Found 11 files for 2018-02-21
Processing [11/11] 2018-02-21-Wednesday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [31]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,web_first)

Found 11 files for 2018-02-22
Processing [11/11] 2018-02-22-Thursday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [32]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,web_second)

Found 11 files for 2018-02-23
Processing [11/11] 2018-02-23-Friday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


In [33]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,infiltration_first)

Found 7 files for 2018-02-28
Processing [7/7] 2018-02-28-Wednesday_TrafficForML_CICFlowMeter_00007.parquet...
Completed splitting 7 files.


In [34]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,infiltration_second)

Found 4 files for 2018-03-01
Processing [4/4] 2018-03-01-Thursday_TrafficForML_CICFlowMeter_00004.parquet...
Completed splitting 4 files.


In [35]:
split_a_single_day(NO_INF_DATA_FOLDER_NAME,SPLIT_DATA_FOLDER_NAME,botnet)

Found 11 files for 2018-03-02
Processing [11/11] 2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet...
Completed splitting 11 files.


Files are successfully split.

### Imputation

imputation for NaN

In [36]:
def get_train_parquet_files(source_folder_name: str) -> list[str]:
    train_files = glob.glob(os.path.join(source_folder_name, "*", "train", "*.parquet"))
    if not train_files:
        raise FileNotFoundError(f"No training Parquet files found in {source_folder_name}")
    print(f"Found {len(train_files)} train files")
    return train_files

In [37]:
def load_combined_lazyframe(train_files: list[str]) -> pl.LazyFrame:
    lazy_frames = [pl.scan_parquet(file) for file in train_files]
    return pl.concat(lazy_frames, how="diagonal_relaxed")

In [38]:
def get_numeric_columns(combined: pl.LazyFrame) -> list[str]:
    schema = combined.collect_schema()
    numeric_cols = [
        column for column, dtype in zip(schema.names(), schema.dtypes())
        if dtype.is_numeric()
    ]
    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [39]:
def compute_medians(combined: pl.LazyFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    for column in numeric_cols:
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs).collect()
    return medians

In [40]:
def build_median_dict(medians: pl.DataFrame, numeric_cols: list[str]) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}")

    return median_dict

In [41]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [42]:
def get_median_imputation(source_folder_name: str, output_file: str) -> dict[str, float]:
    train_files = get_train_parquet_files(source_folder_name)
    combined = load_combined_lazyframe(train_files)
    numeric_cols = get_numeric_columns(combined)
    medians = compute_medians(combined, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [43]:
SPLIT_DATA_FOLDER_NAME = "parquet-split"

In [44]:
median = get_median_imputation(SPLIT_DATA_FOLDER_NAME, "median-imputer.json")
print(json.dumps(median, indent=4))

Found 168 train files
Found 78 numeric columns
Saved medians to: median-imputer.json
{'dst port': 80.0, 'protocol': 6.0, 'flow duration': 20967.0, 'tot fwd pkts': 2.0, 'tot bwd pkts': 1.0, 'totlen fwd pkts': 43.0, 'totlen bwd pkts': 101.0, 'fwd pkt len max': 40.0, 'fwd pkt len min': 0.0, 'fwd pkt len mean': 36.0, 'fwd pkt len std': 0.0, 'bwd pkt len max': 95.0, 'bwd pkt len min': 0.0, 'bwd pkt len mean': 66.0, 'bwd pkt len std': 0.0, 'flow byts/s': 786.0036016321501, 'flow pkts/s': 136.2026696, 'flow iat mean': 11706.0, 'flow iat std': 60.85789458125, 'flow iat max': 18700.0, 'flow iat min': 53.0, 'fwd iat tot': 5147.0, 'fwd iat mean': 3605.0, 'fwd iat std': 0.0, 'fwd iat max': 5037.0, 'fwd iat min': 37.0, 'bwd iat tot': 0.0, 'bwd iat mean': 0.0, 'bwd iat std': 0.0, 'bwd iat max': 0.0, 'bwd iat min': 0.0, 'fwd psh flags': 0.0, 'bwd psh flags': 0.0, 'fwd urg flags': 0.0, 'bwd urg flags': 0.0, 'fwd header len': 40.0, 'bwd header len': 16.0, 'fwd pkts/s': 73.93168712, 'bwd pkts/s': 3.6977

In [45]:
def impute_dataframe(df:pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [46]:
def impute_all_train_files(source_folder_name: str, output_folder_name: str):
    train_files = get_train_parquet_files(source_folder_name)
    total = len(train_files)
    with open("median-imputer.json", "r") as f:
        medians = json.load(f)
    os.makedirs(output_folder_name, exist_ok=True)

    for i, file in enumerate(train_files, start=1):
        df = pd.read_parquet(file)
        df = impute_dataframe(df, medians)
        output_file = os.path.join(output_folder_name, os.path.basename(file))
        df.to_parquet(
            output_file,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        print(f"\r[{i}/{total}] | {os.path.basename(file):<30} ",end="", flush=True)

    print(f"\nFinished imputing {total} files.")

In [47]:
IMPUTED_TRAIN_DATA_FOLDER = "parquet-train-imputed"

In [48]:
impute_all_train_files(SPLIT_DATA_FOLDER_NAME, IMPUTED_TRAIN_DATA_FOLDER)

Found 168 train files
[168/168] | 2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet t  
Finished imputing 168 files.


## Normalized or Feature Scaling

In [ ]:
def create_standard_scaler(source_folder_name: str) -> StandardScaler:
    parquet_files = get_all_file_names(source_folder_name,"parquet")

    if not parquet_files:
        raise FileNotFoundError(f"No Parquet files found in '{source_folder_name}'.")

    scaler = StandardScaler()
    total = len(parquet_files)

    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)

        file_path = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(file_path)

        if df.empty:
            continue

        scaler.partial_fit(df)

    if not hasattr(scaler, "mean_"):
        raise ValueError(
            "The scaler could not be fitted because all files were empty."
        )

    print(" " * 100, end="\r")
    print(f"Successfully fitted scaler using {total} Parquet file(s).")

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler):
    pass

# Principal Component Analysis (PCA)